# Healthcare Knowledge Graph with LangChain

This notebook demonstrates how to query a Neo4j Healthcare Knowledge Graph using LangChain.

## 1. Imports and Environment Setup

In [1]:
from dotenv import load_dotenv
import os
from langchain_community.graphs import Neo4jGraph

load_dotenv()

AURA_INSTANCENAME = os.environ["AURA_INSTANCENAME"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.environ["NEO4J_DATABASE"]
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

## 2. Initialize Neo4j Graph Connection

In [2]:
kg = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

/var/folders/b_/0hx_r16d2v712fg4k7y_rh140000gp/T/ipykernel_14789/3135869052.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(


## 3. Count All Nodes in the Graph

In [3]:
cypher = """
  MATCH (n) 
  RETURN count(n) as numberOfNodes
  """

result = kg.query(cypher)
print(f"There are {result[0]['numberOfNodes']} nodes in this graph.")

There are 115 nodes in this graph.


## 4. Count Healthcare Providers

In [4]:
cypher = """
  MATCH (n:HealthcareProvider) 
  RETURN count(n) AS numberOfProviders
  """
res = kg.query(cypher)
print(f"There are {res[0]['numberOfProviders']} Healthcare Providers in this graph.")

There are 5 Healthcare Providers in this graph.


## 5. Get Healthcare Provider Names

In [5]:
cypher = """
  MATCH (n:HealthcareProvider) 
  RETURN n.name AS ProviderName
  """
res = kg.query(cypher)
print("Healthcare Providers:")
for r in res:
    print(r["ProviderName"])

Healthcare Providers:
Dr. Jessica Lee
Dr. Michael Brown
Dr. Emily Davis
Dr. John Smith
Dr. Sarah Johnson


## 6. List Patients

In [6]:
cypher = """
  MATCH (n:Patient) 
  RETURN n.name AS PatientName
  LIMIT 10
  """
res = kg.query(cypher)
print("Patients:")
for r in res:
    print(r["PatientName"])

Patients:
Eva Blue
Alice Brown
Grace Red
Frank Yellow
David Black
Bob White
Grace Red
Grace Red
Frank Yellow
Frank Yellow


## 7. List Specializations

In [7]:
cypher = """
  MATCH (n:Specialization) 
  RETURN n.name AS SpecializationName
  """
res = kg.query(cypher)
print("Specializations:")
for r in res:
    print(r["SpecializationName"])

Specializations:
Pediatrics
Cardiology
Dermatology
Orthopedics
Neurology


## 8. List Locations

In [8]:
cypher = """
  MATCH (n:Location) 
  RETURN n.name AS LocationName
  """
res = kg.query(cypher)

print("Locations:")
for r in res:
    print(r["LocationName"])

Locations:
Los Angeles
New York
Phoenix
Houston
Chicago


## 9. List Patients Treated by Dr. Smith

In [11]:
cypher = """
  MATCH (hp:HealthcareProvider {name: 'Dr. John Smith'})-[:TREATS]->(p:Patient) 
  RETURN p.name AS PatientName
  """
res = kg.query(cypher)
print("Patients treated by Dr. Smith:")
for r in res:
    print(r["PatientName"])

Patients treated by Dr. Smith:
Eva Blue
Alice Brown
Grace Red
Frank Yellow
David Black
Bob White
Grace Red
Grace Red
Frank Yellow
Frank Yellow
Grace Red
David Black
Alice Brown
Grace Red
Frank Yellow
Eva Blue
Bob White
Alice Brown
Frank Yellow
Charlie Green
Frank Yellow
Bob White
David Black
Bob White
Alice Brown
David Black
Bob White
Charlie Green
Eva Blue
Frank Yellow
Charlie Green
Frank Yellow
Frank Yellow
Frank Yellow
Frank Yellow
Frank Yellow
David Black
Eva Blue
David Black
Grace Red
Alice Brown
David Black
Alice Brown
David Black
Alice Brown
Grace Red
David Black
Eva Blue
Alice Brown
Alice Brown
David Black
Frank Yellow
Frank Yellow
Alice Brown
Frank Yellow
Bob White
David Black
Frank Yellow
Bob White
Eva Blue
Eva Blue
Bob White
Charlie Green
Eva Blue
Grace Red
Bob White
Grace Red
David Black
Grace Red
David Black
Alice Brown
Eva Blue
Alice Brown
David Black
Grace Red
David Black
Charlie Green
Grace Red
Charlie Green
Alice Brown
Eva Blue
Eva Blue
Grace Red
Eva Blue
David Black
C

## 10. List Specializations of Dr. Smith

In [13]:
cypher = """
  MATCH (hp:HealthcareProvider {name: 'Dr. John Smith'})-[:SPECIALIZES_IN]->(s:Specialization) 
  RETURN s.name AS SpecializationName
  """
res = kg.query(cypher)
print("Specializations of Dr. Smith:")
for r in res:
    print(r["SpecializationName"])

Specializations of Dr. Smith:
Pediatrics
Cardiology
Dermatology
Orthopedics
Neurology


## 11. List Healthcare Providers in Houston

In [14]:
cypher = """
  MATCH (hp:HealthcareProvider)-[:LOCATED_AT]->(l:Location {name: 'Houston'}) 
  RETURN hp.name AS ProviderName
  """
res = kg.query(cypher)
print("Healthcare Providers located Houston:")
for r in res:
    print(r["ProviderName"])

Healthcare Providers located Houston:
Dr. Jessica Lee
Dr. Michael Brown
Dr. Emily Davis
Dr. John Smith
Dr. Sarah Johnson


## 12. List Patients Treated by Cardiologists

In [15]:
cypher = """
  MATCH (hp:HealthcareProvider)-[:TREATS]->(p:Patient), 
        (hp)-[:SPECIALIZES_IN]->(s:Specialization {name: 'Cardiology'}) 
  RETURN p.name AS PatientName
  """
res = kg.query(cypher)
print("Patients treated by a Cardiologist:")
for r in res:
    print(r["PatientName"])

Patients treated by a Cardiologist:
Eva Blue
Alice Brown
Grace Red
Frank Yellow
David Black
Bob White
Grace Red
Grace Red
Frank Yellow
Frank Yellow
Grace Red
David Black
Alice Brown
Grace Red
Frank Yellow
Eva Blue
Bob White
Alice Brown
Frank Yellow
Charlie Green
Frank Yellow
Bob White
David Black
Bob White
Alice Brown
David Black
Bob White
Charlie Green
Eva Blue
Frank Yellow
Charlie Green
Frank Yellow
Frank Yellow
Frank Yellow
Frank Yellow
Frank Yellow
David Black
Eva Blue
David Black
Grace Red
Alice Brown
David Black
Alice Brown
David Black
Alice Brown
Grace Red
David Black
Eva Blue
Alice Brown
Alice Brown
David Black
Frank Yellow
Frank Yellow
Alice Brown
Frank Yellow
Bob White
David Black
Frank Yellow
Bob White
Eva Blue
Eva Blue
Bob White
Eva Blue
Grace Red
Bob White
Grace Red
Frank Yellow
David Black
Grace Red
David Black
Eva Blue
David Black
Grace Red
Frank Yellow
David Black
Frank Yellow
Frank Yellow
Grace Red
Frank Yellow
Eva Blue
Eva Blue
Grace Red
Eva Blue
Eva Blue
Bob White
Al

## 13. List Cardiologists in Houston

In [16]:
cypher = """
  MATCH (hp:HealthcareProvider)-[:LOCATED_AT]->(l:Location {name: 'Houston'}), 
        (hp)-[:SPECIALIZES_IN]->(s:Specialization {name: 'Cardiology'}) 
  RETURN hp.name AS ProviderName
  """
res = kg.query(cypher)
print("\nCardiologists located in Houston:")
for r in res:
    print(r["ProviderName"])


Cardiologists located in Houston:
Dr. Jessica Lee
Dr. Michael Brown
Dr. Emily Davis
Dr. John Smith
Dr. Sarah Johnson


## 14. List Patients Treated by Cardiologists in Houston

In [17]:
cypher = """
  MATCH (hp:HealthcareProvider)-[:TREATS]->(p:Patient), 
        (hp)-[:SPECIALIZES_IN]->(s:Specialization {name: 'Cardiology'}), 
        (hp)-[:LOCATED_AT]->(l:Location {name: 'Houston'}) 
  RETURN p.name AS PatientName
  """
res = kg.query(cypher)
print("\nCardiology patients treated by providers in Houston:")
for r in res:
    print(r["PatientName"])


Cardiology patients treated by providers in Houston:
Eva Blue
Alice Brown
Grace Red
Frank Yellow
David Black
Bob White
Grace Red
Grace Red
Frank Yellow
Frank Yellow
Grace Red
David Black
Alice Brown
Grace Red
Frank Yellow
Eva Blue
Bob White
Alice Brown
Frank Yellow
Charlie Green
Frank Yellow
Bob White
David Black
Bob White
Alice Brown
David Black
Bob White
Charlie Green
Eva Blue
Frank Yellow
Charlie Green
Frank Yellow
Frank Yellow
Frank Yellow
Frank Yellow
Frank Yellow
David Black
Eva Blue
David Black
Grace Red
Alice Brown
David Black
Alice Brown
David Black
Alice Brown
Grace Red
David Black
Eva Blue
Alice Brown
Alice Brown
David Black
Frank Yellow
Frank Yellow
Alice Brown
Frank Yellow
Bob White
David Black
Frank Yellow
Bob White
Eva Blue
Eva Blue
Bob White
Eva Blue
Grace Red
Bob White
Grace Red
Frank Yellow
David Black
Grace Red
David Black
Eva Blue
David Black
Grace Red
Frank Yellow
David Black
Frank Yellow
Frank Yellow
Grace Red
Frank Yellow
Eva Blue
Eva Blue
Grace Red
Eva Blue
Eva

## 15. List Patients with Migraine

In [18]:
cypher = """
  MATCH (p:Patient {condition: 'Migraine'}) 
  RETURN p.name AS PatientName
  """
res = kg.query(cypher)
print("\n\n****Patients with Migrane: ***")
for r in res:
    print(r["PatientName"])



****Patients with Migrane: ***
Alice Brown
Frank Yellow
Frank Yellow
Bob White
